In [ ]:
TARGET_NAME = 'BACE1'  # match this to whatever you used in the Part 14 descriptor notebook


In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [ ]:
#@title Import necessary libraries
# If not already installed locally: pip install scikit-learn matplotlib seaborn joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import os

os.makedirs('models', exist_ok=True)
RANDOM_STATE = 42


## 1. Load Data and Define Features/Target


In [ ]:
#@title Auto-fetch descriptor dataset from GitHub (no manual upload needed)
import os, urllib.request
MERGED_URL = "https://raw.githubusercontent.com/anasuyapanyala-ux/BACE1-QSAR-Model/main/BACE1_merged_descriptors.csv"
MERGED_LOCAL = "BACE1_merged_descriptors.csv"
if not os.path.exists(MERGED_LOCAL):
    urllib.request.urlretrieve(MERGED_URL, MERGED_LOCAL)
    print(f"Downloaded {MERGED_LOCAL} from GitHub")
else:
    print(f"{MERGED_LOCAL} already present")

In [ ]:
#@title Load the descriptor dataset for this target
file_path = f'BACE1_merged_descriptors.csv'
data = pd.read_csv(file_path)
print(f"Loaded {len(data)} compounds, {data.shape[1]} columns")
data.head()


In [ ]:
#@title FIX: drop rows with missing Activity instead of imputing the mean
n_before = len(data)
data = data.dropna(subset=['Activity'])
n_after = len(data)
if n_before != n_after:
    print(f"Dropped {n_before - n_after} compounds with missing Activity (not imputed with mean)")

features = data.drop(columns=['SMILES', 'Activity'])
target = data['Activity']

# Any remaining missing feature values (not target) can reasonably be mean-imputed -
# this is standard practice for features, unlike imputing the target/label itself
features = features.fillna(features.mean())

FEATURE_COLUMNS = features.columns.tolist()  # save this - needed later for candidate prediction
print(f"Final training set: {len(features)} compounds, {len(FEATURE_COLUMNS)} features")

# Replace infinite values (can occur in Balaban/Wiener index for certain graph shapes) with NaN, then impute
features = features.replace([np.inf, -np.inf], np.nan)
n_inf = features.isna().sum().sum()
if n_inf:
    print(f"Found and handled {n_inf} infinite/invalid descriptor values")
features = features.fillna(features.median())


## 2. Train/Test Split — THEN Scale (fix for the leakage bug)

Scaling now happens after the split, fit only on training data. The same fitted scaler is reused (never refit) on
the test set and, later, on your candidate compounds.


In [ ]:
#@title Split first...
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=RANDOM_STATE
)


In [ ]:
#@title ...THEN fit the scaler on training data only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)   # fit + transform on train only
X_test = scaler.transform(X_test_raw)         # transform test using train's fitted statistics

joblib.dump(scaler, f'models/{TARGET_NAME}_scaler.pkl')
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")


## 3. Baseline Models (Linear Regression, Random Forest)


In [ ]:
#@title Linear Regression baseline
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)
lr_mse = mean_squared_error(y_test, y_pred_lr)
lr_r2 = r2_score(y_test, y_pred_lr)
print(f"Linear Regression  -  MSE: {lr_mse:.4f}   R2: {lr_r2:.4f}")


In [ ]:
#@title Random Forest baseline
rf_model = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
rf_mse = mean_squared_error(y_test, y_pred_rf)
rf_r2 = r2_score(y_test, y_pred_rf)
print(f"Random Forest  -  MSE: {rf_mse:.4f}   R2: {rf_r2:.4f}")


## 4. Model Comparison with Hyperparameter Tuning

GridSearchCV performs its own internal cross-validation on the training set only — this part of the original
notebook was already leakage-safe, it just inherited the upstream scaling bug. Fixed now since `X_train` is
correctly scaled.


In [ ]:
#@title Define models and hyperparameter grids
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'Random Forest': RandomForestRegressor(random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingRegressor(random_state=RANDOM_STATE),
}

param_grids = {
    'Ridge Regression': {'alpha': [0.1, 1.0, 10.0, 100.0]},
    'Lasso Regression': {'alpha': [0.1, 1.0, 10.0, 100.0]},
    'Random Forest': {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20, 30]},
    'Gradient Boosting': {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 0.2], 'max_depth': [3, 5, 7]},
}


In [ ]:
#@title Evaluate function
def evaluate_model(model_name, model, X_train, X_test, y_train, y_test, param_grid=None):
    if param_grid:
        search = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        print(f"Best parameters for {model_name}: {search.best_params_}")
    else:
        best_model = model
        best_model.fit(X_train, y_train)

    y_pred = best_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"{model_name}  -  MSE: {mse:.4f}   R2: {r2:.4f}")
    return best_model, mse, r2


In [ ]:
#@title Run comparison across all models, track the best by test MSE
best_model = None
best_model_name = None
best_mse = float('inf')
results = []

for model_name, model in models.items():
    param_grid = param_grids.get(model_name, None)
    fitted_model, mse, r2 = evaluate_model(model_name, model, X_train, X_test, y_train, y_test, param_grid)
    results.append({'Model': model_name, 'Mean Squared Error': mse, 'R^2 Score': r2})
    if mse < best_mse:
        best_mse = mse
        best_model = fitted_model
        best_model_name = model_name

print(f"\nBest model: {best_model_name}  (MSE={best_mse:.4f})")

metrics_df = pd.DataFrame(results)
metrics_df.to_csv(f'BACE1_model_performance_metrics.csv', index=False)
metrics_df


## 5. Diagnostics: Actual vs. Predicted, Residuals, Feature Importance


In [ ]:
#@title Actual vs Predicted + Residuals for the best model
y_pred_best = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.scatterplot(x=y_test, y=y_pred_best, alpha=0.7, ax=axes[0])
min_val = min(y_test.min(), y_pred_best.min())
max_val = max(y_test.max(), y_pred_best.max())
axes[0].plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='Ideal Line')
axes[0].set_xlabel('Actual Activity')
axes[0].set_ylabel('Predicted Activity')
axes[0].set_title(f'Actual vs Predicted ({best_model_name})')
axes[0].legend()

residuals = y_test - y_pred_best
sns.histplot(residuals, kde=True, ax=axes[1])
axes[1].set_xlabel('Residuals')
axes[1].set_title('Residuals Distribution')

plt.tight_layout()
plt.savefig(f'BACE1_best_model_diagnostics.png', dpi=150)
plt.show()


In [ ]:
#@title Feature importance (tree-based models only)
if best_model_name in ['Random Forest', 'Gradient Boosting']:
    importances = best_model.feature_importances_
    importance_df = pd.DataFrame({'Feature': FEATURE_COLUMNS, 'Importance': importances})
    importance_df = importance_df.sort_values(by='Importance', ascending=False).head(20)

    plt.figure(figsize=(10, 8))
    sns.barplot(x='Importance', y='Feature', data=importance_df)
    plt.title(f'Top 20 Feature Importances ({best_model_name})')
    plt.tight_layout()
    plt.savefig(f'BACE1_feature_importance.png', dpi=150)
    plt.show()
else:
    print(f"{best_model_name} doesn't expose feature_importances_ (that's a tree-model attribute) - skipping this plot.")


## 6. Save the Best Model


In [ ]:
#@title Save model + the exact feature column order it expects
model_filename = f'models/BACE1_best_model.pkl'
joblib.dump(best_model, model_filename)
joblib.dump(FEATURE_COLUMNS, f'models/BACE1_feature_columns.pkl')
print(f"Saved: {model_filename}  ({best_model_name})")


## 7. Predict on  BBB-Permeable Candidates (FIXED: no silent column mismatch)



In [ ]:
#@title Auto-fetch candidate compounds from GitHub (no manual upload needed)
import os, urllib.request
CAND_URL = "https://raw.githubusercontent.com/anasuyapanyala-ux/BACE1-QSAR-Model/main/candidates_featurized.csv"
CAND_LOCAL = "candidates_featurized.csv"
if not os.path.exists(CAND_LOCAL):
    urllib.request.urlretrieve(CAND_URL, CAND_LOCAL)
    print(f"Downloaded {CAND_LOCAL} from GitHub")
else:
    print(f"{CAND_LOCAL} already present")

In [ ]:
#@title Load candidate descriptors
CANDIDATE_FILE = 'candidates_featurized.csv'  # output of running Part 14's descriptor pipeline on your 113 SMILES
new_data = pd.read_csv(CANDIDATE_FILE)
print(f"Loaded {len(new_data)} candidate compounds")


In [ ]:
#@title FIX: explicit column alignment check before predicting
missing_cols = [c for c in FEATURE_COLUMNS if c not in new_data.columns]
if missing_cols:
    raise ValueError(
        f"Candidate file is missing {len(missing_cols)} column(s) the model expects: {missing_cols}\n"
        f"This means the candidate descriptors weren't generated with the same pipeline as training. "
        f"Re-run the Part 14 descriptor notebook on your candidates before predicting."
    )

# Reindex to guarantee exact same column order the scaler/model were trained on
new_data_features = new_data.reindex(columns=FEATURE_COLUMNS)

# Check for any NaNs introduced by reindexing or already present, and handle explicitly (not silently)
n_nan_rows = new_data_features.isna().any(axis=1).sum()
if n_nan_rows:
    print(f"Warning: {n_nan_rows} candidate(s) have missing feature values - filling with training set means")
    new_data_features = new_data_features.fillna(features.mean())

print("Column alignment check passed - candidate features match training features exactly.")


In [ ]:
#@title Scale (using the SAME scaler fit on training data - never refit here) and predict
new_data_scaled = scaler.transform(new_data_features)
predictions = best_model.predict(new_data_scaled)

new_data[f'{TARGET_NAME}_Predicted_pIC50'] = predictions
output_path = f'BACE1_candidate_predictions.csv'
new_data.to_csv(output_path, index=False)
print(f"Predictions saved to {output_path}")
new_data.sort_values(f'BACE1_Predicted_pIC50', ascending=False)[['SMILES', f'BACE1_Predicted_pIC50']].head(15)
